In [38]:
import git
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
import matplotlib.ticker as ticker
warnings.filterwarnings("ignore")

In [2]:
repo = git.Repo(".", search_parent_directories=True).working_tree_dir
ucd_processed_path = f"{repo}/datasets/raw/Underlying_Cause_of_Death_2022.csv"

In [7]:
cdi_dtype = {
    "Notes": "category",
    "Year": "category",
    "Year Code": "category",
    "Cause of death": "category",
    "Cause of death Code": "category",
    "Deaths": "category",
    "Population": "category",
    "Crude Rate": "category"
}

ucd_df = pd.read_csv(ucd_processed_path, dtype=cdi_dtype, encoding='ISO-8859-1')

ucd_df_selected = ucd_df[['Year', 'Cause of death', 'Cause of death Code', 'Deaths', 'Population', 'Crude Rate']]

heart_related_codes = [
    "I05.0", "I05.1", "I05.2", "I05.8", "I05.9", "I06.0", "I06.9", "I07.1", "I07.8", "I07.9",
    "I08.0", "I08.1", "I08.2", "I08.3", "I08.9", "I09.0", "I09.1", "I09.9", "I10", "I11.0",
    "I11.9", "I12.0", "I12.9", "I13.0", "I13.1", "I13.2", "I13.9", "I20.0", "I20.1", "I20.8",
    "I20.9", "I21.3", "I21.4", "I21.9", "I22.9", "I24.8", "I24.9", "I25.0", "I25.1", "I25.3",
    "I25.5", "I25.8", "I25.9", "I26.0", "I26.9", "I27.0", "I27.1", "I27.2", "I27.8", "I27.9",
    "I28.1", "I28.8", "I28.9", "I30.1", "I30.8", "I30.9", "I31.1", "I31.2", "I31.3", "I31.9",
    "I33.0", "I33.9", "I34.0", "I34.1", "I34.2", "I34.8", "I34.9", "I35.0", "I35.1", "I35.2",
    "I35.8", "I35.9", "I36.1", "I38", "I40.0", "I40.1", "I40.9", "I42.0", "I42.1", "I42.2",
    "I42.4", "I42.5", "I42.6", "I42.7", "I42.8", "I42.9", "I44.1", "I44.2", "I44.3", "I44.7",
    "I45.1", "I45.4", "I45.6", "I45.8", "I45.9", "I46.1", "I46.9", "I47.1", "I47.2", "I48",
    "I49.0", "I49.3", "I49.5", "I49.8", "I49.9", "I50.0", "I50.1", "I50.9", "I51.0", "I51.2",
    "I51.3", "I51.4", "I51.5", "I51.6", "I51.7", "I51.8", "I51.9", "I51A"
]

keywords = ["heart", "cardiovascular", "covid"]

ucd_df_filtered = ucd_df_selected[
    ucd_df_selected['Cause of death Code'].isin(heart_related_codes) |
    ucd_df_selected['Cause of death'].str.contains('|'.join(keywords), case=False, na=False)
]

ucd_df_filtered['Cause of death'] = ucd_df_filtered['Cause of death'].cat.add_categories("Heart Problem")

ucd_df_filtered.loc[ucd_df_filtered['Cause of death Code'].isin(heart_related_codes), 'Cause of death'] = 'Heart Problem'

print(ucd_df_filtered.head())

ucd_df_filtered.to_csv("filtered_heart_cardiovascular_covid_data.csv", index=False)
print("Filtered data saved to 'filtered_heart_cardiovascular_covid_data.csv'")

     Year                                     Cause of death  \
222  2018                        Heart - Malignant neoplasms   
402  2018                           Heart - Benign neoplasms   
804  2018  Rheumatic fever without mention of heart invol...   
805  2018                                      Heart Problem   
806  2018                                      Heart Problem   

    Cause of death Code Deaths Population  Crude Rate  
222               C38.0     60  327167434           0  
402               D15.1     27  327167434           0  
804                 I00     11  327167434  Unreliable  
805               I05.0    577  327167434         0.2  
806               I05.1     41  327167434           0  
Filtered data saved to 'filtered_heart_cardiovascular_covid_data.csv'


In [40]:
repo = git.Repo(".", search_parent_directories=True).working_tree_dir
pds_processed_path = f"{repo}/datasets/raw/Prevalence_of_Disability_Status_and_Types_2022.csv"

# Load the CSV file into a DataFrame
df = pd.read_csv(pds_processed_path)

desired_columns = [
    'Year',
    'LocationAbbr',
    'LocationDesc',
    'Category',
    'Indicator',
    'Response',
    'Data_Value_Type',
    'Data_Value',
    'Stratification1'
]

# Select only the desired columns
df_selected = df[desired_columns]


In [42]:
# Filter for US location and Overall stratification
df_filtered = df_selected[(df_selected['LocationAbbr'] == 'US') & (df_selected['Stratification1'] == 'Overall')]

# Drop the 'Indicator' column, rename 'Category' to 'Topic', 'Response' to 'Question', and 'Number' to 'Data Value'
df_filtered = df_filtered.drop(columns=['Indicator'])
df_filtered = df_filtered.rename(columns={
    'Category': 'Topic',
    'Response': 'Question',
    'Data_Value': 'DataValue',
    'Data_Value_Type':'DataValueType'
})



In [44]:
# Add " Incidence" to the 'Question' column
df_filtered['Question'] = df_filtered['Question'] + ' Prevalence'
#df_filtered['DataValueType'] = df_filtered['DataValueType'].replace("Age-adjusted Prevalence", "Number")


# Drop the 'Data_Value' column if it's in the DataFrame
df_filtered = df_filtered.drop(columns=['Data_Value'], errors='ignore')

# Select and reorder final columns
df_filtered = df_filtered[['Year', 'Topic', 'Question', 'DataValueType', 'DataValue']]

# Save to CSV
df_filtered.to_csv("filtered_Prevalence_of_Disability_Status_and_Types_2022.csv", index=False)

print("Filtered data saved to 'filtered_Prevalence_of_Disability_Status_and_Types_2022.csv'")

Filtered data saved to 'filtered_Prevalence_of_Disability_Status_and_Types_2022.csv'


In [27]:
repo = git.Repo(".", search_parent_directories=True).working_tree_dir
txt_file_path = f"{repo}/datasets/raw/Cancer_Statistics_2021/BYSITE.TXT"  # Replace this with the actual path to your file on GitHub

df = pd.read_csv(txt_file_path, delimiter='|')

# Rename 'RACE' column to 'Stratification1' and 'COUNT' column to 'Deaths'
df = df.rename(columns={'RACE': 'Stratification1', 'COUNT': 'Deaths'})

# Drop the 'SEX' column
df = df.drop(columns=['SEX'])

# Filter for rows where EVENT_TYPE is 'Mortality', Stratification1 is 'All Races', and Year is not '2018-2022'
df_filtered = df[(df['EVENT_TYPE'] == 'Mortality') & 
                 (df['Stratification1'] == 'All Races') & 
                 (df['YEAR'] != '2018-2022')]

# Save the resulting DataFrame to a CSV file
output_csv_path = "filtered_mortality_data.csv"
df_filtered.to_csv(output_csv_path, index=False)

# Display the first few rows of the filtered data
print(df_filtered.head())

   YEAR Stratification1               SITE EVENT_TYPE AGE_ADJUSTED_CI_LOWER  \
1  1999       All Races  Acute Lymphocytic  Mortality                   0.4   
3  2000       All Races  Acute Lymphocytic  Mortality                   0.4   
5  2001       All Races  Acute Lymphocytic  Mortality                   0.4   
7  2002       All Races  Acute Lymphocytic  Mortality                   0.4   
9  2003       All Races  Acute Lymphocytic  Mortality                   0.3   

  AGE_ADJUSTED_CI_UPPER AGE_ADJUSTED_RATE Deaths  POPULATION  
1                   0.4               0.4    582   142237295  
3                   0.4               0.4    591   143719004  
5                   0.5               0.4    644   145077463  
7                   0.4               0.4    616   146394634  
9                   0.4               0.4    574   147679036  
